> **First draft, no longer maintained.** Phase S is being rewritten from zero (see the Phase S page of the course). The material of this notebook moves into the new lessons; this version is kept only for reference.

# S1.4 · Project: the shape of Bibliothecae\.it

*Phase S, statistics and probability · S1, part 4 of 4 · about 3 hours · runs on your data*

Time to use the toolkit of S1.1–S1.3 on real data: the articles and references of [*Bibliothecae.it*](https://bibliothecae.unibo.it/). The result is a short, numbers-backed description of the journal's citation practices: how long its reference lists are, how old the works it cites are, how concentrated its citations are among authors and venues, and how its own authorship is distributed. Such a description is the core of the descriptive section of a data paper.

**Run this lesson on your computer**, where your data are. The Bibliothecae\.it files are not part of the public course repository. If the notebook cannot find them, it runs on a *simulated* stand-in with the same tables and columns, so the code works but the numbers are made up.

**To use your data**, put the seven CSV files from `bib_data_output.zip`, or the zip itself, in the folder `data/bibliothecae/` of your copy of the course. You can also set the environment variable `BIBLIOTHECAE_DATA` to their folder or zip. In JupyterLite or Colab, upload the zip next to this notebook.

> **⚠️ Before you share this notebook.** Once it runs on your data, its outputs contain figures about your dataset. Clear the outputs before committing it to the public repository, or keep your copy in `answers/private/`, which is never committed.

**By the end you have**

- six descriptive analyses of the journal's references and authors, each checked;
- a 2 × 2 figure panel;
- a 250–400 word description of the shape of the journal's citations, with its caveats.

In [ ]:
# ⚙️ Setup: run this cell first
import sys, pathlib, subprocess

if sys.platform == "emscripten":            # in the browser (JupyterLite)
    import piplite
    await piplite.install(["personal-course"])
elif "google.colab" in sys.modules:         # in Google Colab
    repo = "git+https://github.com/sbrzt/personal-course"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"personal-course @ {repo}"], check=True)
else:                                       # on your computer
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:    # find the course folder
        if (folder / "course" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import powerlaw
from course import bibliothecae as bib
from course import data, plots, stats
from course.checks.draft import s1_4 as check

warnings.filterwarnings("ignore", module="powerlaw")
plots.use_style()
print("Ready.")

## 0 · Load and inspect

In [ ]:
corpus = data.bibliothecae()
corpus

In [ ]:
if corpus.demo:
    print("⚠️  SIMULATED DATA: the numbers below are not about Bibliothecae.it.")
corpus.articles.head(3)

The seven tables are linked by identifiers: `articles.id` (Wikidata QIDs), `references.reference` (local IDs such as REF0001) and author IDs, which are Wikidata QIDs for the journal's own authors and local IDs for most cited authors. All values are read as text, so nothing is converted behind your back.

### The rules

Descriptive statistics depend on decisions about the data. These are the rules this project uses, and the checks follow them:

1. Only the articles listed in `articles` count. Relation rows that point to other items are ignored.
2. A **citation** is one link between a citing article and a reference. Exact duplicate rows in the relation tables count once.
3. A **year** is the first four-digit number in a date field.
4. A cited year is **valid** if it lies between 1400 and the year of the citing article.
5. A **venue** is any text between «guillemets» in a reference, with spaces squeezed, outer punctuation stripped and letters lower-cased.

First, a quick look at the problems these rules deal with:

In [ ]:
for name in ["article_author", "article_reference", "reference_author"]:
    table = getattr(corpus, name)
    dups = table.duplicated().sum()
    print(f"{name:18} {len(table):6,} rows, {dups:3} exact duplicates")

years = stats.year_of(corpus.references["date"])
print(f"\nreferences without a year: {years.isna().sum():,} of {len(years):,}")
odd = ((years < 1400) | (years > 2026)).sum()
print(f"references with a year before 1400 or after 2026: {odd:,}")

Write down what you see: these numbers belong in the caveats of your write-up, and they are the starting point of the cleaning work in P1.

The citation links, with the year of the citing article and the cited year:

In [ ]:
articles = corpus.articles.assign(
    citing_year=stats.year_of(corpus.articles["date"]).to_numpy())
refs = corpus.references.assign(
    cited_year=stats.year_of(corpus.references["date"]).to_numpy())

links = (corpus.article_reference.drop_duplicates()             # rule 2
         .merge(articles[["id", "citing_year"]], on="id")       # rule 1
         .merge(refs[["reference", "cited_year", "content"]]
                .drop_duplicates("reference"), on="reference", how="left"))
print(f"{len(links):,} citations from {links['id'].nunique()} articles")
links.head(3)

## 1 · References per article

### ✏️ Exercise 1 · References per article

Compute `refs_per_article`, the number of references of each article that has any (a Series indexed by article id), and `median_refs`, its median.

In [ ]:
refs_per_article = ...
median_refs = ...

check.ex1(corpus, refs_per_article, median_refs)

<details>
<summary>✅ Solution</summary>

```python
refs_per_article = links.groupby("id").size()
median_refs = refs_per_article.median()

check.ex1(corpus, refs_per_article, median_refs)
```

</details>

In [ ]:
r = bib.refs_per_article(corpus)
n_articles = len(corpus.articles)
print(f"{n_articles - r.size} of {n_articles} articles have no references in the data")
stats.summary(r).round(1).to_frame("references per article")

Before interpreting, find out which articles have no references, and why. Are they from particular years, or particular sections (reviews, editorials)? A quick table:

In [ ]:
has_refs = articles["id"].isin(r.index)
table = pd.crosstab(articles["citing_year"].astype("Int64"), has_refs)
table.rename(columns={True: "with refs", False: "without"}).T

## 2 · How old are the cited works?

The **age** of a citation is the citing year minus the cited year. Humanities scholars cite old works, and a journal of bibliography and library history cites very old ones. Two summaries are standard: the median age, and the **Price index**, the share of references at most five years old (Price 1970).

### ✏️ Exercise 2 · How old are the cited works?

Compute `ages` (a Series: one age per citation with a valid cited year, rule 4), `median_age` and `price_idx`.

In [ ]:
valid = ...
ages = ...
median_age = ...
price_idx = ...

check.ex2(corpus, ages, median_age, price_idx)

<details>
<summary>💡 Hint</summary>

A citation is valid when `links["cited_year"]` is at least 1400 and at most `links["citing_year"]`. Comparisons with NaN are `False`, so missing years drop out by themselves.

</details>

<details>
<summary>✅ Solution</summary>

```python
valid = (links["cited_year"] >= 1400) & (links["cited_year"] <= links["citing_year"])
ages = (links["citing_year"] - links["cited_year"])[valid]
median_age = ages.median()
price_idx = np.mean(ages <= 5)

check.ex2(corpus, ages, median_age, price_idx)
```

</details>

In [ ]:
valid = ((links["cited_year"] >= 1400)                           # rule 4
         & (links["cited_year"] <= links["citing_year"]))
ages = (links["citing_year"] - links["cited_year"])[valid]
cited_years = links.loc[valid, "cited_year"]
print(f"valid ages: {ages.size:,}, median {ages.median():.0f} years")
print(f"Price index: {stats.price_index(ages):.2f}")
print(f"cited works older than 1800: {np.mean(cited_years < 1800):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
plots.log_histogram(ages + 1, ax=axes[0], bins=30, color=plots.BLUE)
axes[0].set(xlabel="age of the cited work + 1 (years)", title="Citation age")
by_year = links.assign(age=ages).groupby("citing_year")["age"].median()
axes[1].plot(by_year.index, by_year.values, marker="o", color=plots.BLUE)
axes[1].set(xlabel="year of the citing article", ylabel="median age (years)",
            title="Median citation age by year")
plt.tight_layout()
plt.show()

## 3 · Who gets cited?

### ✏️ Exercise 3 · Who gets cited?

Compute `author_counts`: the number of citations received by each cited author, counting one citation each time a work of theirs is cited. Merge `links` with `corpus.reference_author` on `reference`, then count the authors. Then compute `gini_authors` and `top1_share_authors` (the share of citations going to the top 1% of cited authors).

In [ ]:
author_counts = ...
gini_authors = ...
top1_share_authors = ...

check.ex3(corpus, author_counts, gini_authors, top1_share_authors)

<details>
<summary>✅ Solution</summary>

```python
pairs = links[["id", "reference"]].merge(corpus.reference_author.drop_duplicates(),
                                         on="reference")
author_counts = pairs["author"].value_counts()
gini_authors = stats.gini(author_counts)
top1_share_authors = stats.top_share(author_counts, 0.01)

check.ex3(corpus, author_counts, gini_authors, top1_share_authors)
```

</details>

In [ ]:
counts = bib.cited_author_counts(corpus)
names = bib.author_names(corpus)
top = counts.head(10).rename(index=lambda a: names.get(a, a))
print(f"{counts.size:,} cited authors, {np.mean(counts == 1):.0%} of them cited once")
top.to_frame()

Remember the warning of S1.2: these are authors cited *at least once*. The Gini coefficient describes inequality among them, not among all the authors who could have been cited. Two more caveats belong in your write-up: authors without an identifier, and the same person under several IDs or spellings, both split or merge counts.

## 4 · Power law or lognormal, on your data

### ✏️ Exercise 4 · Power law or lognormal, on your data

Fit `author_counts` with `powerlaw.Fit(..., discrete=True)`, and compare the power law with the lognormal. Report `alpha`, `xmin`, `R` and `p`.

In [ ]:
fit = ...
alpha = ...
xmin = ...
R = ...
p = ...

check.ex4(corpus, alpha, xmin, R, p)

<details>
<summary>✅ Solution</summary>

```python
fit = powerlaw.Fit(counts.to_numpy(), discrete=True, verbose=False)
alpha = fit.power_law.alpha
xmin = fit.power_law.xmin
R, p = fit.distribution_compare("power_law", "lognormal", normalized_ratio=True)

check.ex4(corpus, alpha, xmin, R, p)
```

</details>

Interpret before moving on: which part of the data does the fit describe (how many authors lie above xmin?), and can the data tell the two models apart? Write two sentences in the cell below.

*Your interpretation:*

## 5 · Lotka's law for the journal's authors

### ✏️ Exercise 5 · Lotka's law for the journal's authors

Use `corpus.article_author`, keeping only links to the corpus's articles (rule 1). Compute `table`, the number of authors who wrote *k* articles for each *k* (as in S1.3), and `share_single`.

In [ ]:
own = ...
table = ...
share_single = ...

check.ex5(corpus, table, share_single)

<details>
<summary>✅ Solution</summary>

```python
own = corpus.article_author[corpus.article_author["id"].isin(corpus.articles["id"])]
table = own.drop_duplicates().groupby("author").size().value_counts().sort_index()
share_single = table.loc[1] / table.sum()

check.ex5(corpus, table, share_single)
```

</details>

A journal is a small, edited world, and Lotka's law was formulated for whole fields. Ask what a high share of one-article authors means here (many guest contributors?), and how a few very productive authors (editors, founders) shape the journal. Note too which articles have no author links at all.

## 6 · Bradford's zones for the cited venues

### ✏️ Exercise 6 · Bradford's zones for the cited venues

Extract the venues from `links["content"]` with the pattern `VENUE` and normalise them with `bib.normalise_venue` (rule 5). Count the citations to each venue (`venues`, a Series), then compute the Bradford zones with `stats.bradford_zones(venues, 3)`.

In [ ]:
VENUE = bib.VENUE                              # re.compile(r"«([^»]{2,120})»")
venues = ...
zones = ...

check.ex6(corpus, venues, zones)

<details>
<summary>💡 Hint</summary>

`links["content"].str.findall(VENUE)` gives a list of venues per citation; `.explode()` puts one venue per row; `.dropna().map(bib.normalise_venue)` cleans them; `.value_counts()` counts them.

</details>

<details>
<summary>✅ Solution</summary>

```python
VENUE = bib.VENUE
v = links["content"].str.findall(VENUE).explode().dropna().map(bib.normalise_venue)
venues = v[v != ""].value_counts()
zones = stats.bradford_zones(venues, 3)

check.ex6(corpus, venues, zones)
```

</details>

In [ ]:
venues = bib.venue_counts(corpus)
zones = stats.bradford_zones(venues, 3)
print(f"{venues.sum():,} citations to {venues.size:,} venues")
print(f"venues per Bradford zone: {zones.tolist()}")
print(f"cited once: {np.mean(venues == 1):.0%} of venues")
venues.head(10).to_frame()

If the third zone is much larger than *k*² times the first, the venue list has a long tail of venues cited once: partly real scattering, partly variant spellings of the same venue that the normalisation did not merge. Look at a sample of the venues cited once before deciding which.

In [ ]:
once = venues[venues == 1]
once.sample(min(15, len(once)), random_state=0).index.tolist()

## 7 · A figure panel

One 2 × 2 figure that summarises the shape of the journal's citations. It is saved in a `private/` folder next to this notebook; folders with that name are never committed.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7.5))

axes[0, 0].hist(r, bins=30, color=plots.BLUE, edgecolor="white", linewidth=0.6)
axes[0, 0].set(xlabel="references per article", ylabel="articles",
               title="A · Reference list length")

plots.log_histogram(ages + 1, ax=axes[0, 1], bins=30, color=plots.BLUE)
axes[0, 1].set(xlabel="age of cited work + 1 (years)", title="B · Citation age")

plots.ccdf(counts, ax=axes[1, 0], color=plots.BLUE)
axes[1, 0].set(xlabel="citations received", ylabel="share of cited authors ≥ x",
               title="C · Citations per cited author")

plots.rank_frequency(venues.to_numpy(), ax=axes[1, 1], color=plots.BLUE)
axes[1, 1].set(xlabel="venue rank", ylabel="citations",
               title="D · Citations per venue")

if corpus.demo:
    fig.suptitle("SIMULATED DATA (not Bibliothecae.it)", color=plots.ORANGE,
                 fontweight="bold")
plt.tight_layout()
out = pathlib.Path("private")
out.mkdir(exist_ok=True)
fig.savefig(out / "s1-4-shape-of-bibliothecae.png")
plt.show()

## 8 · Write-up

### ✍️ The shape of Bibliothecae\.it

In 250–400 words, describe the shape of the journal's citations, as you would in the descriptive section of a data paper. Use the numbers from the analyses above, and give medians and percentiles rather than means for skewed quantities. A structure that works:

1. **The corpus.** How many articles, from which years; how many have references, and why the others do not.
2. **Reference lists.** Median and IQR of references per article, and the longest lists.
3. **Age.** Median citation age and Price index; the share of works older than 1800. What this says about the field.
4. **Concentration.** The most cited authors and venues; the Gini coefficient and the top 1% share, saying who was counted; the power law against the lognormal, in one sentence.
5. **Authorship.** The share of single-article authors compared with Lotka's law.
6. **Caveats.** Missing and impossible years, duplicates, unidentified or split authors, venue variants, and the rules you applied.

*Your write-up:*

### Check your progress

In [ ]:
check.summary();

### Done when

- [ ] Exercises 1 to 6 show ✅ in `check.summary()` on your real data (not the simulated stand-in).
- [ ] The figure panel is saved in `private/`.
- [ ] Your write-up gives each number with its rule, and lists the caveats.
- [ ] You have noted the data problems you found, for the cleaning work in P1.

### Further reading

- Price, D. J. de S. (1970). Citation measures of hard science, soft science, technology, and nonscience. In C. E. Nelson & D. K. Pollock (Eds.), *Communication among Scientists and Engineers* (pp. 3–22). Heath.
- Hammarfelt, B. (2016). Beyond coverage: Toward a bibliometrics for the humanities. In M. Ochsner, S. E. Hug & H.-D. Daniel (Eds.), *Research Assessment in the Humanities: Towards Criteria and Procedures*. Springer. <https://doi.org/10.1007/978-3-319-29016-4_10>

**Next:** S2 · Probability and language models.